# SGGF-Net Training Notebook (Google Colab - T4 GPU)

**3-Stage Training Strategy optimized for Colab T4 GPU**

- Stage 1: Baseline Faster-RCNN (8 epochs, ~15-20 min)
- Stage 2: Enable GFEM (6 epochs, ~12-15 min)
- Stage 3: Enable NDPA + ARPM (4 epochs, ~8-10 min)

**Total time: ~35-45 minutes on T4 GPU**

**Setup:**
1. Runtime → Change runtime type → GPU (T4)
2. Run all cells

**Resuming from a Previous Session:**
If your session expired and you want to resume:
- **Resume from Stage 2**: Make sure `stage1_best.pth` is in your Drive at `/content/drive/MyDrive/SGGF-Net-checkpoints/`
- **Resume from Stage 3**: Make sure `stage2_best.pth` is in your Drive
- If using a different Google account, download checkpoints from the previous account's Drive and upload to the new account
- Run the "Check Checkpoints" cell (after mounting Drive) to see what's available

In [ ]:
# Step 1: Mount Drive and clone repository
import os
import subprocess
from google.colab import drive

print("=" * 70)
print("SETUP")
print("=" * 70)

# Mount Drive
try:
    if os.path.exists('/content/drive/MyDrive'):
        print('✓ Google Drive already mounted')
    else:
        drive.mount('/content/drive', force_remount=False)
        print('✓ Google Drive mounted')
except Exception as e:
    print(f'⚠ Drive mounting failed: {e}')
    print('  Continuing without Drive (checkpoints will be saved locally)')

# Clone repository
if not os.path.exists('SGGF-Net'):
    print('\n📦 Cloning repository...')
    result = subprocess.run(['git', 'clone', 'https://github.com/HarishSankarK/SGGF-Net.git'], 
                           capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ Git clone failed: {result.stderr}')
    else:
        print('✓ Repository cloned')
        os.chdir('SGGF-Net')
else:
    print('✓ Repository already exists, using existing')
    os.chdir('SGGF-Net')

# Verify we're in the right place
if not os.path.exists('scripts/train.py'):
    print(f'❌ ERROR: scripts/train.py not found in {os.getcwd()}')
    print('Please check repository structure')
else:
    print(f'\n✓ Working directory: {os.getcwd()}')
    print(f'✓ Found train.py at: {os.path.abspath("scripts/train.py")}')
    
print("=" * 70)

In [ ]:
# Step 2: Install dependencies and verify GPU
import sys
import subprocess

print("=" * 70)
print("INSTALLING DEPENDENCIES")
print("=" * 70)

# Install PyTorch with CUDA
print("Installing PyTorch...")
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu118'], check=False)

# Install other dependencies
print("Installing other dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy', 'pillow', 'opencv-python', 'tqdm', 'matplotlib', 'scipy'], check=False)

# Verify installation
try:
    import torch
    import torchvision
    import numpy
    import PIL
    import cv2
    print(f"\n✓ PyTorch: {torch.__version__}")
    print(f"✓ Torchvision: {torchvision.__version__}")
    
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
        print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
        print(f"✓ CUDA Version: {torch.version.cuda}")
    else:
        print("⚠ No GPU detected! Enable GPU in Runtime settings:")
        print("  Runtime → Change runtime type → GPU (T4)")
        print("  Then re-run this cell")
        
    print("\n✓ All dependencies installed successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please restart runtime and try again")
    
print("=" * 70)

## Check Available Checkpoints (Optional - for Resuming)


In [ ]:
# Check what checkpoints are available
import os

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

print("=" * 70)
print("CHECKING AVAILABLE CHECKPOINTS")
print("=" * 70)

if not os.path.exists(checkpoint_dir):
    print(f"\n⚠ Checkpoint directory not found: {checkpoint_dir}")
    print("  This is normal if you're starting from scratch.")
    print("  Checkpoints will be created after Stage 1 completes.")
else:
    print(f"\n✓ Checkpoint directory found: {checkpoint_dir}")
    
    # Check for stage checkpoints
    stage1_best = os.path.join(checkpoint_dir, 'stage1_best.pth')
    stage1_latest = os.path.join(checkpoint_dir, 'stage1_latest.pth')
    stage2_best = os.path.join(checkpoint_dir, 'stage2_best.pth')
    stage2_latest = os.path.join(checkpoint_dir, 'stage2_latest.pth')
    stage3_best = os.path.join(checkpoint_dir, 'stage3_best.pth')
    stage3_latest = os.path.join(checkpoint_dir, 'stage3_latest.pth')
    
    print("\nStage 1:")
    if os.path.exists(stage1_best):
        size = os.path.getsize(stage1_best) / (1024**2)  # MB
        print(f"  ✓ stage1_best.pth ({size:.1f} MB)")
    else:
        print("  ✗ stage1_best.pth (not found)")
    
    if os.path.exists(stage1_latest):
        size = os.path.getsize(stage1_latest) / (1024**2)  # MB
        print(f"  ✓ stage1_latest.pth ({size:.1f} MB)")
    
    print("\nStage 2:")
    if os.path.exists(stage2_best):
        size = os.path.getsize(stage2_best) / (1024**2)  # MB
        print(f"  ✓ stage2_best.pth ({size:.1f} MB)")
    else:
        print("  ✗ stage2_best.pth (not found)")
    
    if os.path.exists(stage2_latest):
        size = os.path.getsize(stage2_latest) / (1024**2)  # MB
        print(f"  ✓ stage2_latest.pth ({size:.1f} MB)")
    
    print("\nStage 3:")
    if os.path.exists(stage3_best):
        size = os.path.getsize(stage3_best) / (1024**2)  # MB
        print(f"  ✓ stage3_best.pth ({size:.1f} MB)")
    else:
        print("  ✗ stage3_best.pth (not found)")
    
    if os.path.exists(stage3_latest):
        size = os.path.getsize(stage3_latest) / (1024**2)  # MB
        print(f"  ✓ stage3_latest.pth ({size:.1f} MB)")
    
    # Provide resume instructions
    print("\n" + "=" * 70)
    print("RESUME INSTRUCTIONS")
    print("=" * 70)
    
    if os.path.exists(stage1_best) or os.path.exists(stage1_latest):
        print("\n✓ Stage 1 checkpoint found! You can:")
        print("  - Skip Stage 1 and run Stage 2 directly (Cell 6)")
        checkpoint_to_use = stage1_best if os.path.exists(stage1_best) else stage1_latest
        print(f"  - Checkpoint path: {checkpoint_to_use}")
    
    if os.path.exists(stage2_best) or os.path.exists(stage2_latest):
        print("\n✓ Stage 2 checkpoint found! You can:")
        print("  - Skip Stages 1 & 2 and run Stage 3 directly (Cell 8)")
        checkpoint_to_use = stage2_best if os.path.exists(stage2_best) else stage2_latest
        print(f"  - Checkpoint path: {checkpoint_to_use}")
    
    if os.path.exists(stage3_best) or os.path.exists(stage3_latest):
        print("\n✓ Stage 3 checkpoint found! Training is complete.")
        print("  - You can run Evaluation (Cell 10)")
        checkpoint_to_use = stage3_best if os.path.exists(stage3_best) else stage3_latest
        print(f"  - Checkpoint path: {checkpoint_to_use}")
    
    if not any([os.path.exists(p) for p in [stage1_best, stage1_latest, stage2_best, stage2_latest, stage3_best, stage3_latest]]):
        print("\n⚠ No checkpoints found. Start from Stage 1 (Cell 4)")

print("\n" + "=" * 70)


## Stage 1: Baseline Faster-RCNN


## Stage 1: Baseline Faster-RCNN

In [ ]:
# Stage 1 Training
import subprocess
import os
import sys

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Verify we're in the right directory
if not os.path.exists('scripts/train.py'):
    print('❌ ERROR: scripts/train.py not found!')
    print(f'Current directory: {os.getcwd()}')
    print('Please make sure Cell 1 cloned the repository correctly.')
    sys.exit(1)

# Verify dataset exists
if not os.path.exists('data/hit-uav'):
    print('⚠ Dataset not found at data/hit-uav')
    print('The dataset needs to be downloaded. Checking if it exists elsewhere...')
    # Check if dataset might be in a different location
    possible_paths = [
        '/content/drive/MyDrive/datasets/hit-uav',
        '/content/hit-uav',
        'hit-uav'
    ]
    found = False
    for path in possible_paths:
        if os.path.exists(path):
            print(f'✓ Found dataset at: {path}')
            # Create symlink or copy
            os.makedirs('data', exist_ok=True)
            if not os.path.exists('data/hit-uav'):
                os.symlink(os.path.abspath(path), 'data/hit-uav')
                print(f'✓ Created symlink: data/hit-uav -> {path}')
                found = True
                break
    
    if not found:
        print('\n❌ Dataset not found. Please download HIT-UAV dataset:')
        print('  1. Go to: https://github.com/Syo9/HIT-UAV')
        print('  2. Download and extract to: data/hit-uav/')
        print('  3. Or upload to Google Drive and update the path above')
        print('\nExpected structure:')
        print('  data/hit-uav/')
        print('    ├── images/')
        print('    └── annotations/')
        sys.exit(1)
else:
    print(f'✓ Dataset found at: {os.path.abspath("data/hit-uav")}')

print("=" * 70)
print("STAGE 1: BASELINE FASTER-RCNN")
print("=" * 70)
print("Starting training... (output will stream below)")
print("⚠ First batch may take 30-60 seconds to compile on GPU\n")

cmd = ['python', '-u', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', checkpoint_dir, '--stage', '1', '--subset_ratio', '0.35']

# Run with real-time output streaming (unbuffered)
# Use Popen to stream output line by line
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Merge stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
    universal_newlines=True
)

# Stream output in real-time
for line in process.stdout:
    print(line, end='', flush=True)

# Wait for completion
process.wait()

if process.returncode != 0:
    print(f"\n❌ Training failed with exit code {process.returncode}")
    sys.exit(1)
else:
    print("\n✓ Stage 1 training completed successfully!")

## Stage 2: Enable GFEM

In [ ]:
# Stage 2 Training
import subprocess
import os
import sys

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Try to find Stage 1 checkpoint (best or latest)
stage1_best = os.path.join(checkpoint_dir, 'stage1_best.pth')
stage1_latest = os.path.join(checkpoint_dir, 'stage1_latest.pth')

stage1_checkpoint = None
if os.path.exists(stage1_best):
    stage1_checkpoint = stage1_best
    print(f"✓ Found Stage 1 best checkpoint: {stage1_best}")
elif os.path.exists(stage1_latest):
    stage1_checkpoint = stage1_latest
    print(f"✓ Found Stage 1 latest checkpoint: {stage1_latest}")
else:
    print(f"⚠ Stage 1 checkpoint not found in: {checkpoint_dir}")
    print("\n📋 TO RESUME FROM STAGE 2:")
    print("=" * 70)
    print("Option 1: Upload checkpoint from your computer")
    print("  1. Download stage1_best.pth from your previous account's Google Drive")
    print("  2. In Colab: Files → Upload to session storage")
    print("  3. Upload stage1_best.pth")
    print("  4. Run this command in a new cell:")
    print(f"     !mkdir -p {checkpoint_dir} && cp /content/stage1_best.pth {checkpoint_dir}/")
    print("\nOption 2: If checkpoint is in a different Drive location")
    print("  - Modify the checkpoint_dir variable above")
    print("  - Or manually copy: !cp /path/to/stage1_best.pth " + checkpoint_dir)
    print("\nOption 3: Re-run Stage 1")
    print("  - Go back to Cell 7 (Stage 1) and run it")
    print("=" * 70)
    sys.exit(1)

print("=" * 70)
print("STAGE 2: ENABLE GFEM")
print("=" * 70)
print("Starting training... (output will stream below)")
print("⚠ First batch may take 30-60 seconds to compile on GPU\n")

cmd = ['python', '-u', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', checkpoint_dir, '--stage', '2', '--resume', stage1_checkpoint, '--subset_ratio', '0.35']

# Run with real-time output streaming (unbuffered)
# Use Popen to stream output line by line
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Merge stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
    universal_newlines=True
)

# Stream output in real-time
for line in process.stdout:
    print(line, end='', flush=True)

# Wait for completion
process.wait()

if process.returncode != 0:
    print(f"\n❌ Training failed with exit code {process.returncode}")
    sys.exit(1)
else:
    print("\n✓ Stage 2 training completed successfully!")

## Stage 3: Enable NDPA + ARPM

In [ ]:
# Stage 3 Training
import subprocess
import os
import sys
import shutil

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Debug: List all files in checkpoint directory
print("=" * 70)
print("CHECKING FOR STAGE 2 CHECKPOINT")
print("=" * 70)
print(f"Checkpoint directory: {checkpoint_dir}")
if os.path.exists(checkpoint_dir):
    files = os.listdir(checkpoint_dir)
    print(f"\nFiles in checkpoint directory ({len(files)} files):")
    for f in sorted(files):
        if f.endswith('.pth'):
            size = os.path.getsize(os.path.join(checkpoint_dir, f)) / (1024**2)  # MB
            print(f"  - {f} ({size:.1f} MB)")
    if not any(f.endswith('.pth') for f in files):
        print("  (no .pth files found)")
else:
    print(f"⚠ Checkpoint directory does not exist: {checkpoint_dir}")

# Try to find Stage 2 checkpoint (best or latest)
stage2_best = os.path.join(checkpoint_dir, 'stage2_best.pth')
stage2_latest = os.path.join(checkpoint_dir, 'stage2_latest.pth')

stage2_checkpoint = None
if os.path.exists(stage2_best):
    stage2_checkpoint = stage2_best
    print(f"\n✓ Found Stage 2 best checkpoint: {stage2_best}")
elif os.path.exists(stage2_latest):
    stage2_checkpoint = stage2_latest
    print(f"\n✓ Found Stage 2 latest checkpoint: {stage2_latest}")
    # Create best from latest if best doesn't exist (for convenience)
    if not os.path.exists(stage2_best):
        try:
            shutil.copy2(stage2_latest, stage2_best)
            print(f"✓ Created stage2_best.pth from latest checkpoint")
        except Exception as e:
            print(f"⚠ Could not create best from latest: {e}")
else:
    print(f"\n⚠ Stage 2 checkpoint not found in: {checkpoint_dir}")
    
    # Fallback: Try to create Stage 2 checkpoint from Stage 1
    stage1_best = os.path.join(checkpoint_dir, 'stage1_best.pth')
    stage1_latest = os.path.join(checkpoint_dir, 'stage1_latest.pth')
    
    stage1_checkpoint = None
    if os.path.exists(stage1_best):
        stage1_checkpoint = stage1_best
    elif os.path.exists(stage1_latest):
        stage1_checkpoint = stage1_latest
    
    if stage1_checkpoint:
        print("\n" + "=" * 70)
        print("⚠ FALLBACK: Creating Stage 2 checkpoint from Stage 1")
        print("=" * 70)
        print("⚠ WARNING: Stage 2 training did not complete properly!")
        print("  This means GFEM was not trained. Stage 3 will use Stage 1 weights.")
        print("  Results may be suboptimal. It's recommended to re-run Stage 2 first.")
        print("\n  To proceed with fallback:")
        print(f"  - Creating stage2_best.pth from {os.path.basename(stage1_checkpoint)}")
        
        try:
            import torch
            # Load Stage 1 checkpoint
            checkpoint = torch.load(stage1_checkpoint, map_location='cpu', weights_only=False)
            # Update checkpoint metadata for Stage 2
            checkpoint['stage'] = 2
            checkpoint['epoch'] = 0  # Reset epoch for new stage
            # Save as Stage 2 checkpoint
            stage2_fallback = os.path.join(checkpoint_dir, 'stage2_best.pth')
            torch.save(checkpoint, stage2_fallback)
            stage2_checkpoint = stage2_fallback
            print(f"  ✓ Created fallback checkpoint: {stage2_fallback}")
            print("\n  ⚠ You can proceed, but consider re-running Stage 2 for better results.")
            print("=" * 70)
        except Exception as e:
            print(f"  ❌ Failed to create fallback checkpoint: {e}")
            print("\n📋 MANUAL OPTIONS:")
            print("=" * 70)
            print("Option 1: Re-run Stage 2 (Recommended)")
            print("  - Go back to Cell 9 (Stage 2) and run it")
            print("  - Make sure to pull latest code: git pull origin main")
            print("\nOption 2: Upload checkpoint manually")
            print("  1. Download stage2_best.pth or stage2_latest.pth from Google Drive")
            print("  2. In Colab: Files → Upload to session storage")
            print("  3. Upload the checkpoint file")
            print("  4. Run this command in a new cell:")
            print(f"     !mkdir -p {checkpoint_dir}")
            print(f"     !cp /content/stage2_*.pth {checkpoint_dir}/")
            print("=" * 70)
            sys.exit(1)
    else:
        print("\n❌ No Stage 1 checkpoint found either!")
        print("\n📋 TO RESUME FROM STAGE 3:")
        print("=" * 70)
        print("Option 1: Re-run Stage 2 (Recommended)")
        print("  - Go back to Cell 9 (Stage 2) and run it")
        print("  - Make sure to pull latest code: git pull origin main")
        print("\nOption 2: Upload checkpoint manually")
        print("  1. Download stage2_best.pth or stage2_latest.pth from Google Drive")
        print("  2. In Colab: Files → Upload to session storage")
        print("  3. Upload the checkpoint file")
        print("  4. Run this command in a new cell:")
        print(f"     !mkdir -p {checkpoint_dir}")
        print(f"     !cp /content/stage2_*.pth {checkpoint_dir}/")
        print("=" * 70)
        sys.exit(1)

print("=" * 70)
print("STAGE 3: ENABLE NDPA + ARPM")
print("=" * 70)
print("Starting training... (output will stream below)")
print("⚠ First batch may take 30-60 seconds to compile on GPU\n")

cmd = ['python', '-u', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', checkpoint_dir, '--stage', '3', '--resume', stage2_checkpoint, '--subset_ratio', '0.35']

# Run with real-time output streaming (unbuffered)
# Use Popen to stream output line by line
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Merge stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
    universal_newlines=True
)

# Stream output in real-time
for line in process.stdout:
    print(line, end='', flush=True)

# Wait for completion
process.wait()

if process.returncode != 0:
    print(f"\n❌ Training failed with exit code {process.returncode}")
    sys.exit(1)
else:
    print("\n✓ Stage 3 training completed successfully!")

## Evaluate Final Model

In [ ]:
# Evaluation with Visualization
import subprocess
import os
import sys
import torch
from IPython.display import Image, display
import matplotlib.pyplot as plt

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
results_dir = 'results'
os.makedirs(results_dir, exist_ok=True)

final_checkpoint = os.path.join(checkpoint_dir, 'stage3_best.pth')
if not os.path.exists(final_checkpoint):
    final_checkpoint = os.path.join(checkpoint_dir, 'stage3_latest.pth')

if not os.path.exists(final_checkpoint):
    print(f"⚠ {final_checkpoint} not found. Complete all stages first!")
    sys.exit(1)

device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
print("=" * 70)
print("EVALUATION")
print("=" * 70)
print("Starting evaluation... (output will stream below)\n")

cmd = ['python', '-u', 'scripts/evaluate.py', '--dataset', 'hituav', '--data_dir', 'data/hit-uav', '--checkpoint', final_checkpoint, '--num_classes', '6', '--batch_size', '1', '--max_size', '640', '--split', 'test', '--device', device_str, '--output_dir', results_dir]

# Run with real-time output streaming (unbuffered)
# Use Popen to stream output line by line
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Merge stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
    universal_newlines=True
)

# Stream output in real-time
for line in process.stdout:
    print(line, end='', flush=True)

# Wait for completion
process.wait()

if process.returncode != 0:
    print(f"\n❌ Evaluation failed with exit code {process.returncode}")
    sys.exit(1)
else:
    print("\n✓ Evaluation completed successfully!")
    
    # Display visualization graphs
    print("\n" + "=" * 70)
    print("VISUALIZATION GRAPHS")
    print("=" * 70)
    
    graph_files = [
        os.path.join(results_dir, 'overall_metrics.png'),
        os.path.join(results_dir, 'per_class_ap.png'),
        os.path.join(results_dir, 'metrics_distribution.png')
    ]
    
    graph_titles = [
        "Overall Evaluation Metrics",
        "Per-Class Average Precision (AP@0.5)",
        "Metrics Distribution"
    ]
    
    for graph_file, title in zip(graph_files, graph_titles):
        if os.path.exists(graph_file):
            print(f"\n📊 {title}:")
            display(Image(graph_file))
        else:
            print(f"\n⚠ Graph not found: {graph_file}")
    
    print("\n" + "=" * 70)
    print(f"✓ All graphs saved to: {os.path.abspath(results_dir)}/")
    print("=" * 70)